In [22]:
import ROOT
import numpy as np
import math

In [4]:
def GetInterpolation(x_values,y_values,desired_point):
    graph = ROOT.TGraph(len(x_values))
    for i in range(len(x_values)):
        graph.SetPoint(i,x_values[i],y_values[i])
    #graph.Draw()
    #raw_input("bla")
    interpol = graph.Eval(desired_point)
    #print interpol
    return interpol

In [16]:
mphi = np.array([ 995, 1495, 1250, 1000, 2495,  200,  200, 2000,  2000, 1995,
         195, 1750, 1750, 1700, 1500, 1500,  750,  500,  495,  3000,
         2995, 2500, 2500,  300,  500])

y  = np.array([ 
    47.1250, 4.0938, 1.7812 , 642.0000  , 0.0522,
    4.8281, 36.2500,35.8125, 176.7500,0.4941,25.6250,
    17.1250, 28.3750,8.9688, 4000.0000,
    0.6504, 239.5000, 1.3711, 36.3750
    
    
    , 519.0, 0.7266, 177.5, 
    0.0942, 0.2393, 8.4062,
         
         ])

In [17]:
GetInterpolation(mphi, y, np.array(900))

121.71938775510205

In [23]:
def CalculateBR(mphi,mchi,g_dm=1.0,g_quarks=0.25,method=0):
    if mchi >= mphi/2.:
        raise ValueError("mchi must be smaller than mphi/2")
    width_dm = None
    if method == 0:
        width_dm = mphi/(24*math.pi)*math.sqrt(1-4*(mchi**2)/(mphi**2))*(g_dm**2)*(2*(1-(mchi**2)/(mphi**2))+6*(mchi**2)/(mphi**2))
    elif method == 1:
        width_dm = (g_dm**2)*mphi/(12*math.pi)*math.sqrt(1-4*(mchi**2)/(mphi**2))*(1+2*(mchi**2)/(mphi**2))
    width_quarks = mphi/(4*math.pi)*(g_quarks**2)*(1-(172.5**2)/(mphi**2))*(1-(172.5**2)/(2*(mphi**2))-(172.5**4)/(2*(mphi**4)))+mphi/(4*math.pi)*(g_quarks**2)
    br = width_dm/(width_dm+width_quarks)
    # try:
    #     print "brancing fraction: ",br
    # except ValueError:
    #     print "bla"
    #     exit()
    return br

In [25]:
CalculateBR(200, 99)

0.5070272006013566

In [ ]:
import ROOT
import sys
import os
import csv
from array import array
import math
import optparse

from pprint import pprint

ROOT.gROOT.SetBatch(True)
ROOT.gStyle.SetOptStat(0)


# change color map
NCont         = 255
red_   = [1.0, 0.5, 0.0, 1.0]
green_ = [0.0, 1.0, 1.0, 1.0]
blue_  = [0.0, 0.0, 1.0, 1.0]
stops_ = [0.0, 0.3, 0.6, 1.0]
NRGBs  = len(red_)

red = array('d', red_)
green = array('d', green_)
blue = array('d', blue_)
stops = array('d', stops_)

# from https://github.com/LPC-DM/METXFit/blob/SuperMonojetFit/plotting/plotLimits2D.py#L150-L164
# ncontours = 999;
# stops_ = [ 0.0000,       0.10,   0.200,     0.30,    0.4000,    0.50,    0.7500,    0.8750, 1.0000]
# red_   = [ 243./255., 243./255., 240./255., 240./255., 241./255., 239./255., 186./255., 151./255., 129./255.]
# green_ = [   0./255.,  46./255.,  99./255., 149./255., 194./255., 220./255., 183./255., 166./255., 147./255.]
# blue_  = [   6./255.,   8./255.,  36./255.,  91./255., 169./255., 235./255., 246./255., 240./255., 233./255.]

stopsArray = array('d',stops)
redArray   = array('d',red)
greenArray = array('d',green)
blueArray  = array('d',blue)

ROOT.TColor.CreateGradientColorTable(9, stops, red, green, blue, NCont)
ROOT.gStyle.SetNumberContours(NCont)
# ROOT.TColor.InvertPalette()

script_path = os.path.dirname(os.path.realpath(__file__))

def GetLimit(file_,type="expected"):
    file = ROOT.TFile.Open(file_)
    tree = file.Get("limit")
    if type=="expected":
        tree.GetEntry(2)
    elif type=="observed":
        tree.GetEntry(5)
    elif type=="down1":
        tree.GetEntry(1)
    elif type=="down2":
        tree.GetEntry(0)
    elif type=="up1":
        tree.GetEntry(3)
    elif type=="up2":
        tree.GetEntry(4)
    else:
        print "which kind of limit?"
        exit()
    limit = tree.limit
    file.Close()
    return limit

def find_masses(dataset_name):
    #dataset_name = dataset_name.replace("-","_")
    dataset_name = dataset_name.lower()
    index_1 = dataset_name.find("mphi_")
    index_2 = dataset_name.find("mchi_")
    #print(index_1,index_2)
    number_1 = ""
    number_2 = ""
    for char in dataset_name[index_1+5:]:
        #print(char)
        if not char.isdigit(): break
        number_1+=char
    for char in dataset_name[index_2+5:]:
        #print(char)
        if not char.isdigit(): break
        number_2+=char
    print(dataset_name)
    print("Mphi="+number_1+" Mchi="+number_2)
    return float(number_1),float(number_2)

def FindChannel(filename):
    channel = None
    if "leptonic" in filename:
        channel = "leptonic"
    elif "hadronic" in filename:
        channel = "hadronic"
    elif "combined" in filename:
        channel = "combined"
    else:
        print("channel not recognized")
        exit()
    return channel

def FindYear(era):
    year = None
    if "2016preVFP" in era:
        year = "19.52 fb^{-1} (13 TeV), 2016preVFP"
    elif "2016postVFP" in era:
        year = "16.81 fb^{-1} (13 TeV), 2016postVFP"
    elif "2016" in era:
        year = "36.33 fb^{-1} (13 TeV), 2016"
    elif "2017" in era:
        year = "41.48 fb^{-1} (13 TeV), 2017"
    elif "2018" in era:
        year = "59.83 fb^{-1} (13 TeV), 2018"
    elif "1718" in era:
        year = "101.31 fb^{-1} (13 TeV), 2017-2018"
    elif "all_years" in era:
        year = "137.64 fb^{-1} (13 TeV), 2016-2018"
    else:
        print("year not recognized")
        exit()
    return year

def ReadXSDict():
    xs_dict={}
    p = os.path.join(script_path,"monotop_signals_xs.csv")
    with open(p) as csvfile:
        reader = csv.DictReader(csvfile)
        for row in reader:
            xs = float(row["xs"])
            # xs_dict[row["name"]]="{:5.12f}".format(xs).rstrip("0")
            xs_dict[row["name"]]="{}".format(xs).rstrip("0")
    pprint(xs_dict)
    return xs_dict

def FindXS(xs_dict,datacard_name):
    for key in xs_dict:
        print(key)
        if key in datacard_name:
            return xs_dict[key]
    # return None
    # return 1.0

def GetInterpolation(x_values,y_values,desired_point):
    graph = ROOT.TGraph(len(x_values))
    for i in range(len(x_values)):
        graph.SetPoint(i,x_values[i],y_values[i])
    #graph.Draw()
    #raw_input("bla")
    interpol = graph.Eval(desired_point)
    #print interpol
    return interpol

def CalculateBR(mphi,mchi,g_dm=1.0,g_quarks=0.25,method=0):
    if mchi >= mphi/2.:
        raise ValueError("mchi must be smaller than mphi/2")
    width_dm = None
    if method == 0:
        width_dm = mphi/(24*math.pi)*math.sqrt(1-4*(mchi**2)/(mphi**2))*(g_dm**2)*(2*(1-(mchi**2)/(mphi**2))+6*(mchi**2)/(mphi**2))
    elif method == 1:
        width_dm = (g_dm**2)*mphi/(12*math.pi)*math.sqrt(1-4*(mchi**2)/(mphi**2))*(1+2*(mchi**2)/(mphi**2))
    width_quarks = mphi/(4*math.pi)*(g_quarks**2)*(1-(172.5**2)/(mphi**2))*(1-(172.5**2)/(2*(mphi**2))-(172.5**4)/(2*(mphi**4)))+mphi/(4*math.pi)*(g_quarks**2)
    br = width_dm/(width_dm+width_quarks)
    # try:
    #     print "brancing fraction: ",br
    # except ValueError:
    #     print "bla"
    #     exit()
    return br


parser = optparse.OptionParser()
parser.add_option("--outdir","-o",
    dest="outdir",
    help="output directory for limit plot (default: '.')",
    default = ".")
parser.add_option("--outLabel",
    dest="outLabel",
    help="additional label for output filename (default=None)",
    default = None)
parser.add_option("--channelLabel","-l",
    dest="channelLabel",
    help="channelLabel on plot (default: 'leptonic')",
    default = "leptonic")
parser.add_option("--era","-e",
    dest="era",
    help="set era, only used for lumilabel at the moment (default: '2018')",
    default = "2018")
parser.add_option("--unblind","-u",
    dest="unblind",
    help="also draw observed limits (default: False)",
    default = False,
    action = "store_true")
parser.add_option("--extrapolate",
    dest="extrapolate",
    help="extrapolate to additional DM masses (default: False)",
    default = False,
    action = "store_true")

(options, files) = parser.parse_args()

cross_section_dict = ReadXSDict()

print(files)
graph2Dexp = ROOT.TGraph2D()
graph2Dup1 = ROOT.TGraph2D()
graph2Ddown1 = ROOT.TGraph2D()
graph2Dup2 = ROOT.TGraph2D()
graph2Ddown2 = ROOT.TGraph2D()
graph2Dobs = ROOT.TGraph2D()
graph2Dobs_th_up = ROOT.TGraph2D()
graph2Dobs_th_down = ROOT.TGraph2D()

results_dict = {}
results_dict_extra={}
issues = [] 
for i in range(len(files)):
    print(FindXS(cross_section_dict,files[i]))
    xs = float(FindXS(cross_section_dict,files[i]))
    masses = find_masses(files[i])
    print "cross section: ",xs
    # xs = 1.
    explimit = GetLimit(files[i],"expected")/xs
    if explimit == 0:
        print("explimit is 0")
        print(masses)
        issues.append(masses)
        continue
    up1 = GetLimit(files[i],"up1")/xs
    down1 = GetLimit(files[i],"down1")/xs
    up2 = GetLimit(files[i],"up2")/xs
    down2 = GetLimit(files[i],"down2")/xs
    obslimit = GetLimit(files[i],"observed")/xs
    obslimit_th_up = GetLimit(files[i],"observed")/(xs*1.1)
    obslimit_th_down = GetLimit(files[i],"observed")/(xs*0.9)
    if not masses[0] in results_dict:
        results_dict[masses[0]]={}
    if not masses[1] in results_dict[masses[0]]:
        results_dict[masses[0]][masses[1]]={}

    if not masses[0] in results_dict_extra:
        results_dict_extra[masses[0]]={}
    if not masses[1] in results_dict_extra[masses[0]]:
        results_dict_extra[masses[0]][masses[1]]={}

    
    results_dict[masses[0]][masses[1]]["exp"]=explimit
    results_dict[masses[0]][masses[1]]["expup1"]=up1
    results_dict[masses[0]][masses[1]]["expup2"]=up2
    results_dict[masses[0]][masses[1]]["expdown1"]=down1
    results_dict[masses[0]][masses[1]]["expdown2"]=down2
    results_dict[masses[0]][masses[1]]["obs"]=obslimit
    results_dict[masses[0]][masses[1]]["obsthup"]=obslimit_th_up
    results_dict[masses[0]][masses[1]]["obsthdown"]=obslimit_th_down
    print "mphi,mchi ",masses
    print "explimit ",explimit
print("original grid:")
pprint(results_dict)
print(issues)
# exit()

# do interpolations here 
graphmasspoints_extrapolated = ROOT.TGraph()
graphmasspoints_extrapolated.SetName("graphmasspoints")

stepsize = 25
extrapolations = {}
for mchi in [200, 300, 500, 750, 1000, 1250, 1500, 1750, 2000, 2500, 3000]:
    extrapolations[mchi] = range(0, mchi/2, stepsize)
    if mchi == 1000:
        extrapolations[mchi].append(499)

if options.extrapolate:
    i = 0
    for mphi in extrapolations.keys():
        if not mphi in results_dict.keys():
            continue
        for newmchi in extrapolations[mphi]:
            anchor = 150.
            if mphi == 300:
                anchor = 100.
            if mphi == 200:
                anchor = 50.
            if mphi == 2500:
                anchor = 750.
            if mphi == 3000:
                anchor = 1000.
            if newmchi == anchor:
                continue
            if not anchor in results_dict[mphi].keys():
                continue
            i+=1
            try:
                print("extrapolating to ({},{}) from ({},{})".format(mphi,newmchi,mphi,anchor))
                results_dict[mphi][newmchi]={}
                results_dict[mphi][newmchi]["exp"]=results_dict[mphi][anchor]["exp"]*1./(CalculateBR(mphi,newmchi)/CalculateBR(mphi,anchor))
                results_dict[mphi][newmchi]["expup1"]=results_dict[mphi][anchor]["expup1"]*1./(CalculateBR(mphi,newmchi)/CalculateBR(mphi,anchor))
                results_dict[mphi][newmchi]["expup2"]=results_dict[mphi][anchor]["expup2"]*1./(CalculateBR(mphi,newmchi)/CalculateBR(mphi,anchor))
                results_dict[mphi][newmchi]["expdown1"]=results_dict[mphi][anchor]["expdown1"]*1./(CalculateBR(mphi,newmchi)/CalculateBR(mphi,anchor))
                results_dict[mphi][newmchi]["expdown2"]=results_dict[mphi][anchor]["expdown2"]*1./(CalculateBR(mphi,newmchi)/CalculateBR(mphi,anchor))
                results_dict[mphi][newmchi]["obs"]=results_dict[mphi][anchor]["obs"]*1./(CalculateBR(mphi,newmchi)/CalculateBR(mphi,anchor))
                results_dict[mphi][newmchi]["obsthup"]=results_dict[mphi][anchor]["obsthup"]*1./(CalculateBR(mphi,newmchi)/CalculateBR(mphi,anchor))
                results_dict[mphi][newmchi]["obsthdown"]=results_dict[mphi][anchor]["obsthdown"]*1./(CalculateBR(mphi,newmchi)/CalculateBR(mphi,anchor))
                graphmasspoints_extrapolated.SetPoint(i,mphi,newmchi)

                results_dict_extra[mphi][newmchi]={}
                results_dict_extra[mphi][newmchi]["exp"]=results_dict[mphi][anchor]["exp"]*1./(CalculateBR(mphi,newmchi)/CalculateBR(mphi,anchor))
                results_dict_extra[mphi][newmchi]["expup1"]=results_dict[mphi][anchor]["expup1"]*1./(CalculateBR(mphi,newmchi)/CalculateBR(mphi,anchor))
                results_dict_extra[mphi][newmchi]["expup2"]=results_dict[mphi][anchor]["expup2"]*1./(CalculateBR(mphi,newmchi)/CalculateBR(mphi,anchor))
                results_dict_extra[mphi][newmchi]["expdown1"]=results_dict[mphi][anchor]["expdown1"]*1./(CalculateBR(mphi,newmchi)/CalculateBR(mphi,anchor))
                results_dict_extra[mphi][newmchi]["expdown2"]=results_dict[mphi][anchor]["expdown2"]*1./(CalculateBR(mphi,newmchi)/CalculateBR(mphi,anchor))
            except:
                continue



# print("-----real-----")
# print("995,500")
# pprint(results_dict[995][500])
# print("-----extra-----")
# print("1000,499")
# pprint(results_dict_extra[1000][499])
# exit()

# construct graphs
graphmasspoints = ROOT.TGraph()
i = 0
for mphi in results_dict.keys():
    for mchi in results_dict[mphi].keys():
        i+=1
        graphmasspoints.SetPoint(i,mphi,mchi)
        graph2Dexp.SetPoint(i,mphi,mchi,results_dict[mphi][mchi]["exp"])
        graph2Dup1.SetPoint(i,mphi,mchi,results_dict[mphi][mchi]["expup1"])
        graph2Ddown1.SetPoint(i,mphi,mchi,results_dict[mphi][mchi]["expdown1"])
        graph2Dup2.SetPoint(i,mphi,mchi,results_dict[mphi][mchi]["expup2"])
        graph2Ddown2.SetPoint(i,mphi,mchi,results_dict[mphi][mchi]["expdown2"])
        graph2Dobs.SetPoint(i,mphi,mchi,results_dict[mphi][mchi]["obs"])
        graph2Dobs_th_up.SetPoint(i,mphi,mchi,results_dict[mphi][mchi]["obsthup"])
        graph2Dobs_th_down.SetPoint(i,mphi,mchi,results_dict[mphi][mchi]["obsthdown"])

maxChi = 1300
minChi = 50
maxPhi = 2500
minPhi = 20.
maxMu  = 200
# set some dummy limits at borders
for gr in [graph2Dexp,graph2Dup1, graph2Ddown1, graph2Dup2, graph2Ddown2, graph2Dobs, graph2Dobs_th_up, graph2Dobs_th_down]:
    gr.SetPoint(i+1,minPhi,minChi,0) #links unten
    gr.SetPoint(i+2,maxPhi,minChi,maxMu) #rechts unten
    gr.SetPoint(i+3,minPhi,maxChi,maxMu) #links oben
    gr.SetPoint(i+4,maxPhi,maxChi,maxMu) #rechts oben

# set some dummy limits in unallowed region
j = 1
stepsize = 100
gr_unallowed = ROOT.TGraph()
for mphi in [0]+range(100,maxPhi,stepsize):
    for mchi in range(int((mphi)/2+250),maxChi+stepsize,stepsize):
        for gr in [graph2Dexp,graph2Dup1, graph2Ddown1, graph2Dup2, graph2Ddown2, graph2Dobs, graph2Dobs_th_up, graph2Dobs_th_down]:
            gr.SetPoint(i+j,mphi,mchi,maxMu)
        j+=1
        gr_unallowed.SetPoint(j,mphi,mchi)

# create canvas
ROOT.gStyle.SetPadRightMargin(0.15)
ROOT.gStyle.SetPadBottomMargin(0.12)
w=1200
h=800
canvas = ROOT.TCanvas()
canvas.SetCanvasSize(w,h)
canvas.SetLogz()
ROOT.gPad.SetTickx()
ROOT.gPad.SetTicky()
contours = array('d',[1.0])

h = ROOT.TH2D("h","h",1000,minPhi,maxPhi,1000,minChi,maxChi)
h.GetXaxis().SetTitle("M_{V} [GeV]")
h.GetXaxis().SetTitleSize(0.05)
h.GetXaxis().SetLabelSize(0.04)

h.GetYaxis().SetTitle("M_{#chi} [GeV]")
h.GetYaxis().SetTitleSize(0.05)
h.GetYaxis().SetTitleOffset(1.0)
h.GetYaxis().SetLabelSize(0.04)

h.GetZaxis().SetTitle("#mu_{UL}^{95% CL}")
h.GetZaxis().SetTitleSize(0.05)
h.GetZaxis().SetTitleOffset(0.9)
h.GetZaxis().SetRangeUser(0.01,maxMu)

graph2Dexp.SetHistogram(h)
graph2Dexp.SetTitle("")

graph2D_copy = graph2Dexp.Clone()
h_copy = h.Clone()
h_copy.SetContour(1,contours)
graph2D_copy.SetHistogram(h_copy)
graph2D_copy.SetLineWidth(2)

graph2Dup1.GetHistogram().SetContour(1,contours)
graph2Ddown1.GetHistogram().SetContour(1,contours)
graph2Dup2.GetHistogram().SetContour(1,contours)
graph2Ddown2.GetHistogram().SetContour(1,contours)
graph2Dobs.GetHistogram().SetContour(1,contours)
graph2Dobs_th_up.GetHistogram().SetContour(1,contours)
graph2Dobs_th_down.GetHistogram().SetContour(1,contours)

graph2Dobs.SetLineColor(ROOT.kRed)
graph2Dobs.SetLineWidth(2)
graph2Dobs_th_up.SetLineColor(ROOT.kRed)
graph2Dobs_th_up.SetLineWidth(2)
graph2Dobs_th_up.SetLineStyle(4)
graph2Dobs_th_down.SetLineColor(ROOT.kRed)
graph2Dobs_th_down.SetLineWidth(2)
graph2Dobs_th_down.SetLineStyle(4)
graph2Dup1.SetLineStyle(4)
graph2Dup1.SetLineWidth(2)
graph2Ddown1.SetLineStyle(4)
graph2Ddown1.SetLineWidth(2)
graph2Dup2.SetLineStyle(3)
graph2Dup2.SetLineWidth(2)
graph2Ddown2.SetLineStyle(3)
graph2Ddown2.SetLineWidth(2)

# start drawing stuff
graph2Dexp.Draw("colz")
graph2D_copy.Draw("cont3 same")
graph2Dup1.Draw("cont3 same")
graph2Ddown1.Draw("cont3 same")
# graph2Dup2.Draw("cont3 same")
# graph2Ddown2.Draw("cont3 same")
if options.unblind:
    graph2Dobs.Draw("cont3 same")
    graph2Dobs_th_up.Draw("cont3 same")
    graph2Dobs_th_down.Draw("cont3 same")


graphmasspoints.SetMarkerSize(3)
graphmasspoints.SetMarkerStyle(3)
graphmasspoints.Draw("P")

gr_unallowed.SetMarkerSize(3)
gr_unallowed.SetMarkerStyle(3)
gr_unallowed.SetMarkerColor(4)
# gr_unallowed.Draw("P")

if options.extrapolate:
    graphmasspoints_extrapolated.SetMarkerSize(3)
    graphmasspoints_extrapolated.SetMarkerStyle(3)
    graphmasspoints_extrapolated.SetMarkerColor(2)
    graphmasspoints_extrapolated.Draw("P")


legend = ROOT.TLegend(0.15,0.53,0.42,0.88)
legend.SetHeader("#splitline{non-resonant mono-top}{g_{V,q}=0.25  g_{V,#chi}=1.0}")
legend.AddEntry(graph2D_copy,"Median expected","l")
legend.AddEntry(graph2Dup1,"Exp. #pm 1#sigma","l")
# legend.AddEntry(graph2Dup2,"Exp. #pm 2#sigma","l")
if options.unblind:
    legend.AddEntry(graph2Dobs,"Observed","l")
    legend.AddEntry(graph2Dobs_th_up,"Obs. #pm 1#sigma","l")
legend.SetTextSize(0.035)

# labels
# cms = ROOT.TLatex(0.13, 0.93, '#splitline{CMS}{#scale[0.7]{work in progress}}'  )
cms = ROOT.TLatex(0.13, 0.93, 'CMS#scale[0.7]{ work in progress}'  )
cms.SetNDC()
cms.SetTextSize(0.055)
# ROOT.gStyle.SetPalette(1)
# ROOT.gStyle.SetPalette(53)
lumi = ROOT.TLatex(0.85, 0.91, "{0}, {1}".format(options.channelLabel, FindYear(options.era)))
lumi.SetNDC()
lumi.SetTextSize(0.035)
lumi.SetTextAlign(31)

cms.Draw("same")
lumi.Draw("same")
legend.Draw("same")
name = "limit"
if options.unblind:
    name+="_observed"
else:
    name+="_expectedOnly"

if not os.path.exists(options.outdir):
    os.makedirs(options.outdir)

if options.extrapolate:
    name+="_extrapolated"
if options.outLabel:
    name+="_"+options.outLabel
canvas.Print(options.outdir+"/"+ name+"_{}.pdf".format(options.era))
canvas.Print(options.outdir+"/"+ name+"_{}.png".format(options.era))

In [22]:
stepsize = 25
extrapolations = {}
for mchi in [200., 300., 500., 750., 1000., 1250., 1500., 1750., 2000., 2500., 3000.]:
    extrapolations[mchi] = range(0, int(mchi/2), stepsize)
    if mchi == 1000:
        extrapolations[mchi].append(499)

AttributeError: 'range' object has no attribute 'append'

In [21]:
extrapolations

{200.0: range(0, 100, 25),
 300.0: range(0, 150, 25),
 500.0: range(0, 250, 25),
 750.0: range(0, 375, 25),
 1000.0: range(0, 500, 25)}

In [16]:
range(0, int(1000/2), 25)

range(0, 500, 25)

In [14]:
range(0, 500, 25)

range(0, 500, 25)

In [ ]:
print("expected", GetLimit("higgsCombineTest.AsymptoticLimits.mH125.root","expected"))
print("down1", GetLimit("higgsCombineTest.AsymptoticLimits.mH125.root","down1"))
print("down2", GetLimit("higgsCombineTest.AsymptoticLimits.mH125.root","down2"))
print("up1", GetLimit("higgsCombineTest.AsymptoticLimits.mH125.root","up1"))
print("up2", GetLimit("higgsCombineTest.AsymptoticLimits.mH125.root","up2"))